# 03 · EDA y split train/val/test anti-fuga

**Camanchaca-Predict — G5 · Proyecto Aplicado 2026-2**

1. EDA del dataset etiquetado: tamaños, muestras por banda, duplicados.
2. **Split 70/15/15 agrupado por escena** — el punto metodológico crítico:
   RESIDE genera varias nieblas de la MISMA escena; si una va a train y otra
   a test, el modelo memoriza la escena y las métricas se inflan (fuga).
3. Verificación de ausencia de fuga + balance de bandas por conjunto.
4. Exporta `data/splits/{train,val,test}.csv` y `class_weights.json`.

Requiere: `labels.csv` (notebook 02).

In [ ]:
# Setup estándar del proyecto
import random
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

SEED = 42
random.seed(SEED); np.random.seed(SEED)

try:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive/camanchaca")
except ImportError:
    ROOT = Path("local_workspace")

DATA_DIR = ROOT / "data"
LABELS_PATH = DATA_DIR / "processed" / "labels.csv"
SPLITS_DIR = DATA_DIR / "splits"
SPLITS_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3,
                     "axes.spines.top": False, "axes.spines.right": False})

assert LABELS_PATH.exists(), "Ejecuta primero el notebook 02 (falta labels.csv)"
labels = pd.read_csv(LABELS_PATH)
BAND_NAMES = ["critico", "alto_riesgo", "precaucion", "aceptable"]
BAND2IDX = {b: i for i, b in enumerate(BAND_NAMES)}
print(f"{len(labels)} imágenes · {labels['scene'].nunique()} escenas · "
      f"{labels['beta'].nunique()} betas distintos")

In [ ]:
# Chequeos básicos de calidad
print("Duplicados por (scene, beta):", labels.duplicated(subset=["scene", "beta"]).sum())
print("NaN:", labels.isna().sum().sum())
print("Rutas existentes:", labels["path"].map(lambda p: Path(p).exists()).mean())

tamanos = labels["path"].sample(min(100, len(labels)), random_state=SEED)\
                        .map(lambda p: Image.open(p).size)
print("Tamaños de imagen más comunes:", pd.Series(tamanos).value_counts().head(3).to_dict())
print("Decisión: resize+crop a 224x224 (notebooks 04/05).")

In [ ]:
# Galería: 4 ejemplos por banda (con su V real)
n_por_banda = min(4, int(labels["band"].value_counts().min()))
fig, axes = plt.subplots(4, n_por_banda, figsize=(3.2 * n_por_banda, 12),
                         constrained_layout=True, squeeze=False)
for bi, band in enumerate(BAND_NAMES):
    sub = labels[labels["band"] == band]
    sub = sub.sample(min(n_por_banda, len(sub)), random_state=SEED)
    for k in range(n_por_banda):
        ax = axes[bi, k] if n_por_banda > 1 else axes[bi]
        if k < len(sub):
            r = sub.iloc[k]
            ax.imshow(Image.open(r["path"]))
            ax.set_title(f"{r['visibility_m']:.0f} m", fontsize=10)
        ax.axis("off")
    axes[bi, 0].text(-0.08, 0.5, band, transform=axes[bi, 0].transAxes,
                     rotation=90, va="center", ha="center", fontsize=11, fontweight="bold")
fig.suptitle("Ejemplos por banda de seguridad (V real en el título)", fontsize=13)
plt.show()

## Split agrupado por escena

La unidad de split es la **escena**, no la imagen. Entre 50 permutaciones
aleatorias de escenas elegimos la de menor sesgo en la distribución de
bandas. Implementación idéntica a `src/camanchaca/data/splits.py`
(verificada por `tests/test_splits.py`).

In [ ]:
# Split agrupado por escena (anti-fuga) con búsqueda de mínimo sesgo de bandas
def _assign_scenes(counts, order, n_total, test_frac, val_frac):
    test_s, val_s, train_s = [], [], []
    n_test = n_val = 0
    for s in order:
        c = counts[s]
        if n_test < test_frac * n_total:
            test_s.append(s); n_test += c
        elif n_val < val_frac * n_total:
            val_s.append(s); n_val += c
        else:
            train_s.append(s)
    return train_s, val_s, test_s

def grouped_split(df, val_frac=0.15, test_frac=0.15, n_seeds=50, seed=SEED):
    counts = df.groupby("scene").size().to_dict()
    n_total = len(df)
    rng_master = np.random.RandomState(seed)
    best = None
    for _ in range(n_seeds):
        rng = np.random.RandomState(rng_master.randint(0, 2**31 - 1))
        order = rng.permutation(list(counts.keys()))
        tr_s, va_s, te_s = _assign_scenes(counts, order, n_total, test_frac, val_frac)
        tr = df[df["scene"].isin(tr_s)]
        va = df[df["scene"].isin(va_s)]
        te = df[df["scene"].isin(te_s)]
        glob = df["band"].value_counts(normalize=True)
        worst = max(float((d["band"].value_counts(normalize=True)
                           .reindex(glob.index, fill_value=0.0) - glob).abs().max())
                    for d in (tr, va, te) if len(d) > 0)
        if best is None or worst < best[0]:
            best = (worst, tr, va, te)
    _, tr, va, te = best
    return (tr.reset_index(drop=True), va.reset_index(drop=True), te.reset_index(drop=True))

train_df, val_df, test_df = grouped_split(labels)
print(f"train={len(train_df)} ({len(train_df)/len(labels):.0%}) · "
      f"val={len(val_df)} ({len(val_df)/len(labels):.0%}) · "
      f"test={len(test_df)} ({len(test_df)/len(labels):.0%})")

In [ ]:
# VERIFICACIÓN ANTI-FUGA (debe dar 0 en las tres líneas)
print("Escenas train ∩ val :", len(set(train_df["scene"]) & set(val_df["scene"])))
print("Escenas train ∩ test:", len(set(train_df["scene"]) & set(test_df["scene"])))
print("Escenas val   ∩ test:", len(set(val_df["scene"]) & set(test_df["scene"])))
assert (set(train_df["scene"]) & set(test_df["scene"])) == set(), 'FUGA DETECTADA'

In [ ]:
# Balance de bandas por conjunto (debe ser similar entre barras)
fig, ax = plt.subplots(figsize=(9, 4), constrained_layout=True)
x = np.arange(len(BAND_NAMES)); w = 0.26
for off, (nombre, d) in enumerate([("train", train_df), ("val", val_df), ("test", test_df)]):
    props = d["band"].value_counts(normalize=True).reindex(BAND_NAMES, fill_value=0.0)
    ax.bar(x + (off - 1) * w, props.values, w, label=f"{nombre} (n={len(d)})")
ax.set_xticks(x); ax.set_xticklabels(BAND_NAMES)
ax.set_ylabel("proporción"); ax.set_title("Distribución de bandas por conjunto")
ax.legend()
plt.show()

In [ ]:
# Target de regresión: estandarización de log10(V) con estadísticas de TRAIN
MU = float(train_df["log10_v"].mean())
SIGMA = float(train_df["log10_v"].std())

for d in (train_df, val_df, test_df):
    d["y_norm"] = (d["log10_v"] - MU) / SIGMA
    d["band_idx"] = d["band"].map(BAND2IDX).astype(int)
print(f"MU={MU:.4f}  SIGMA={SIGMA:.4f}  (se reutilizan en notebooks 04-06)")

# Pesos de clase para la pérdida CE del ViT (mitigación R3: desbalance)
freq = train_df["band"].value_counts().reindex(BAND_NAMES).fillna(1)
class_weights = (len(train_df) / (len(BAND_NAMES) * freq)).round(3)
print("class_weights:", class_weights.to_dict())

In [ ]:
# Exportamos splits + estadísticas (Drive + repo local si existe)
import json

train_df.to_csv(SPLITS_DIR / "train.csv", index=False)
val_df.to_csv(SPLITS_DIR / "val.csv", index=False)
test_df.to_csv(SPLITS_DIR / "test.csv", index=False)
with open(SPLITS_DIR / "class_weights.json", "w") as f:
    json.dump({"mu": MU, "sigma": SIGMA,
               "class_weights": class_weights.to_dict()}, f, indent=2)

REPO_LOCAL = Path("/content/camanchaca-predict")
if REPO_LOCAL.exists():
    dest = REPO_LOCAL / "data" / "splits"
    dest.mkdir(parents=True, exist_ok=True)
    train_df.to_csv(dest / "train.csv", index=False)
    val_df.to_csv(dest / "val.csv", index=False)
    test_df.to_csv(dest / "test.csv", index=False)
    import shutil
    shutil.copy(SPLITS_DIR / "class_weights.json", dest / "class_weights.json")
    print("Splits copiados también al repo (se commitean: son CSV pequeños)")

print(f"Splits guardados en {SPLITS_DIR}")
display(train_df[["image", "scene", "beta", "visibility_m", "band", "y_norm"]].head())

## Resumen del notebook

- ✅ Split 70/15/15 **agrupado por escena**, semilla 42, mínimo sesgo de bandas.
- ✅ Verificación anti-fuga (0 escenas compartidas) — evidencia para la rúbrica.
- ✅ `y_norm` estandarizado con estadísticas de train (MU/SIGMA guardadas).
- ✅ `class_weights.json` para el ViT (mitigación R3).

**Siguiente:** notebook 04 · baseline_resnet50 (entrenamiento del baseline
exigido por la rúbrica).